# Assignment 6 — Logistic Regression on House Price Data

### Important note
`price` is a **continuous numerical variable**, so Logistic Regression cannot directly use the raw `price` as a classification target. Also, `stratify=y` requires class labels.

Therefore, for this assignment, we create a binary classification target:
- `1` = price is at or above the median price
- `0` = price is below the median price

The original `price` column is kept unchanged in the dataset and excluded from the feature matrix to avoid target leakage.


## 1. Load and explore the dataset

In [1]:
import pandas as pd
import numpy as np

# If the CSV is uploaded to Colab, update this path if necessary
file_path = "/content/Housing.csv"

df = pd.read_csv(file_path)

print("First 5 rows:")
display(df.head())

print("\nDataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nBasic statistics:")
display(df.describe())


First 5 rows:


,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished



Dataset shape: (545, 13)

Column names:
['price', 'area', 'bedrooms', 'bathrooms', 'stories', 'mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'parking', 'prefarea', 'furnishingstatus']

Data types:
price                int64
area                 int64
bedrooms             int64
bathrooms            int64
stories              int64
mainroad            object
guestroom           object
basement            object
hotwaterheating     object
airconditioning     object
parking              int64
prefarea            object
furnishingstatus    object
dtype: object

Missing values:
price               0
area                0
bedrooms            0
bathrooms           0
stories             0
mainroad            0
guestroom           0
basement            0
hotwaterheating     0
airconditioning     0
parking             0
prefarea            0
furnishingstatus    0
dtype: int64

Basic statistics:


,price,area,bedrooms,bathrooms,stories,parking
count,5.450000e+02,545.000000,545.000000,545.000000,545.000000,545.000000
mean,4.766729e+06,5150.541284,2.965138,1.286239,1.805505,0.693578
std,1.870440e+06,2170.141023,0.738064,0.502470,0.867492,0.861586
min,1.750000e+06,1650.000000,1.000000,1.000000,1.000000,0.000000
25%,3.430000e+06,3600.000000,2.000000,1.000000,1.000000,0.000000
50%,4.340000e+06,4600.000000,3.000000,1.000000,2.000000,0.000000
75%,5.740000e+06,6360.000000,3.000000,2.000000,2.000000,1.000000
max,1.330000e+07,16200.000000,6.000000,4.000000,4.000000,3.000000


## 2. Separate X (features) and y (target)

Because Logistic Regression is a classification algorithm, convert the continuous `price` into a binary target using the median price.


In [2]:
# Create binary classification target from price
price_median = df["price"].median()

df["price_class"] = (df["price"] >= price_median).astype(int)

# X = features, y = classification target
X = df.drop(columns=["price", "price_class"])
y = df["price_class"]

print("Median price:", price_median)
print("\nTarget distribution:")
print(y.value_counts().sort_index())


Median price: 4340000.0

Target distribution:
price_class
0    270
1    275
Name: count, dtype: int64


## 3. Label Encode

Label encode:
`mainroad`, `guestroom`, `basement`, `hotwaterheating`, `prefarea`, `airconditioning`


In [3]:
from sklearn.preprocessing import LabelEncoder

label_cols = [
    "mainroad",
    "guestroom",
    "basement",
    "hotwaterheating",
    "prefarea",
    "airconditioning"
]

label_encoders = {}

for col in label_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    label_encoders[col] = le

print("Label encoding completed.")
display(X.head())


Label encoding completed.


,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,7420,4,2,3,1,0,0,0,1,2,1,furnished
1,8960,4,4,4,1,0,0,0,1,3,0,furnished
2,9960,3,2,2,1,0,1,0,0,2,1,semi-furnished
3,7500,4,2,2,1,0,1,0,1,3,1,furnished
4,7420,4,1,2,1,1,1,0,1,2,0,furnished


## 4. One-Hot Encode

One-hot encode `furnishingstatus`.


In [4]:
X = pd.get_dummies(
    X,
    columns=["furnishingstatus"],
    dtype=int
)

print("One-hot encoding completed.")
print("\nFeatures after encoding:")
print(X.columns.tolist())

display(X.head())


One-hot encoding completed.

Features after encoding:
['area', 'bedrooms', 'bathrooms', 'stories', 'mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'parking', 'prefarea', 'furnishingstatus_furnished', 'furnishingstatus_semi-furnished', 'furnishingstatus_unfurnished']


,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus_furnished,furnishingstatus_semi-furnished,furnishingstatus_unfurnished
0,7420,4,2,3,1,0,0,0,1,2,1,1,0,0
1,8960,4,4,4,1,0,0,0,1,3,0,1,0,0
2,9960,3,2,2,1,0,1,0,0,2,1,0,1,0
3,7500,4,2,2,1,0,1,0,1,3,1,1,0,0
4,7420,4,1,2,1,1,1,0,1,2,0,1,0,0


## 5. Train-test split

Use:
- 80% training data
- 20% testing data
- `stratify=y` to preserve the class distribution


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True))


X_train: (436, 14)
X_test : (109, 14)
y_train: (436,)
y_test : (109,)

Training target distribution:
price_class
1    0.504587
0    0.495413
Name: proportion, dtype: float64


## 6. Scale the numeric features with StandardScaler

Numeric features:
`area`, `bedrooms`, `bathrooms`, `stories`, `parking`

Fit the scaler only on the training data, then transform both training and test data.


In [6]:
from sklearn.preprocessing import StandardScaler

numeric_cols = [
    "area",
    "bedrooms",
    "bathrooms",
    "stories",
    "parking"
]

scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_cols] = scaler.fit_transform(
    X_train[numeric_cols]
)

X_test_scaled[numeric_cols] = scaler.transform(
    X_test[numeric_cols]
)

print("Numeric features scaled successfully.")
display(X_train_scaled.head())


Numeric features scaled successfully.


,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus_furnished,furnishingstatus_semi-furnished,furnishingstatus_unfurnished
486,0.415344,-1.299122,-0.573076,-0.906518,1,0,0,0,0,-0.832391,0,0,1,0
452,1.849043,0.075677,-0.573076,0.216328,1,0,0,0,0,1.486602,0,0,1,0
311,0.444018,-1.299122,-0.573076,-0.906518,1,0,1,0,0,0.327106,0,0,1,0
487,0.128604,1.450476,-0.573076,0.216328,1,0,0,0,0,-0.832391,0,0,0,1
297,-0.712499,0.075677,1.378963,0.216328,1,0,1,0,0,-0.832391,0,0,0,1


## 7. Logistic Regression + 5-fold and 10-fold Cross Validation

Train a Logistic Regression model and report:
1. Test accuracy
2. 5-fold cross-validation accuracy
3. 10-fold cross-validation accuracy


In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score

model = LogisticRegression(
    max_iter=2000,
    random_state=42
)

# Train the model
model.fit(X_train_scaled, y_train)

# Test accuracy
y_pred = model.predict(X_test_scaled)
test_accuracy = accuracy_score(y_test, y_pred)

# 5-fold cross-validation
cv5 = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv5_scores = cross_val_score(
    model,
    X_train_scaled,
    y_train,
    cv=cv5,
    scoring="accuracy"
)

# 10-fold cross-validation
cv10 = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

cv10_scores = cross_val_score(
    model,
    X_train_scaled,
    y_train,
    cv=cv10,
    scoring="accuracy"
)

print("========== RESULTS ==========")
print(f"Test Accuracy       : {test_accuracy:.4f}")
print(f"5-Fold CV Mean      : {cv5_scores.mean():.4f}")
print(f"5-Fold CV Std       : {cv5_scores.std():.4f}")
print(f"10-Fold CV Mean     : {cv10_scores.mean():.4f}")
print(f"10-Fold CV Std      : {cv10_scores.std():.4f}")

print("\n5-Fold scores:")
print(cv5_scores)

print("\n10-Fold scores:")
print(cv10_scores)


========== RESULTS ==========
Test Accuracy       : 0.8716
5-Fold CV Mean      : 0.8281
5-Fold CV Std       : 0.0320
10-Fold CV Mean     : 0.8281
10-Fold CV Std      : 0.0456

5-Fold scores:
[0.79545455 0.8045977  0.83908046 0.88505747 0.81609195]

10-Fold scores:
[0.75       0.84090909 0.79545455 0.88636364 0.84090909 0.79545455
 0.86046512 0.90697674 0.79069767 0.81395349]


## Final Observation

The model predicts whether a house belongs to the **higher-price group** or **lower-price group** based on the available housing features.

> Note: If the instructor strictly requires raw `price` as `y`, then Logistic Regression is not appropriate because raw house price is a regression target. In that case, Linear Regression (or another regression model) should be used instead.
